# EfficientNet — Rethinking Model Scaling for Convolutional Neural Networks (2019)

_Papers · Computer Vision_

**Scale a network's depth, width, and input resolution together with one dial φ instead of one at a time.**

---

This notebook is a practice scaffold for the **EfficientNet — Rethinking Model Scaling for Convolutional Neural Networks (2019)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# ===================================================================
# 0. The compound-scaling rule (Eqn. 3) + the worked example.
# ===================================================================
ALPHA, BETA, GAMMA = 1.2, 1.1, 1.15            # paper's grid-search result (sec.4)
print("constraint  alpha*beta^2*gamma^2 =",
      round(ALPHA * BETA**2 * GAMMA**2, 4), "  (target ~2)")

def scale(phi):
    """Eqn. 3: depth d=alpha^phi, width w=beta^phi, resolution r=gamma^phi."""
    d = ALPHA ** phi
    w = BETA  ** phi
    r = GAMMA ** phi
    return d, w, r

d, w, r = scale(2)                              # the lesson's worked example, phi=2
print("phi=2 ->  d =", round(d,4), " w =", round(w,4), " r =", round(r,4))
print("  -> ~%d%% more layers, ~%d%% more channels, ~%d%% bigger per side"
      % (round((d-1)*100), round((w-1)*100), round((r-1)*100)))
flops_mult = d * w**2 * r**2                    # depth linear, width & resolution quadratic
print("  predicted FLOPS multiplier d*w^2*r^2 =", round(flops_mult,3),
      " vs 2^phi =", 2**2)
# constraint alpha*beta^2*gamma^2 = 1.9203 ; phi=2 -> d=1.44 w=1.21 r=1.3225 ; FLOPS~3.687


# ===================================================================
# 1. A tiny baseline CNN (our stand-in for "B0", phi=0).
#    Per-stage (out_channels, num_layers); width multiplies channels,
#    depth multiplies num_layers, resolution sets the input size.
# ===================================================================
BASE_STAGES = [(24, 2), (48, 2), (96, 3)]       # (channels, layers) at phi=0
BASE_RES    = 48                                 # baseline input side (px)

def round_ch(c, w):                              # keep channels a multiple of 8
    return max(8, int(round(c * w / 8)) * 8)

def build(phi=0.0, depth_only=False, res_only=False, width_only=False):
    d, w, r = scale(phi)
    if depth_only:  d, w, r = (ALPHA*BETA**2*GAMMA**2)**phi, 1.0, 1.0   # all budget -> depth
    if width_only:  w = ((ALPHA*BETA**2*GAMMA**2)**phi) ** 0.5; d, r = 1.0, 1.0
    if res_only:    r = ((ALPHA*BETA**2*GAMMA**2)**phi) ** 0.5; d, w = 1.0, 1.0
    layers, in_ch = [], 3
    for ch, L in BASE_STAGES:
        out_ch  = round_ch(ch, w)
        n_layer = max(1, int(round(L * d)))      # depth multiplier, rounded to whole layers
        for j in range(n_layer):
            stride = 2 if j == 0 else 1          # downsample once per stage
            layers += [nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
                       nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
            in_ch = out_ch
    net = nn.Sequential(*layers, nn.AdaptiveAvgPool2d(1),
                        nn.Flatten(), nn.Linear(in_ch, 10))
    res = max(8, int(round(BASE_RES * r)))
    return net, res

# --- A few-line FLOPS counter for Conv2d + Linear (multiply-adds). ---
def count(net, res):
    flops, params = 0, sum(p.numel() for p in net.parameters())
    x = torch.zeros(1, 3, res, res)
    hooks = []
    def conv_hook(m, inp, out):
        nonlocal flops
        oh, ow = out.shape[2], out.shape[3]
        flops += m.in_channels * m.out_channels * m.kernel_size[0] * m.kernel_size[1] * oh * ow
    def lin_hook(m, inp, out):
        nonlocal flops
        flops += m.in_features * m.out_features
    for m in net.modules():
        if isinstance(m, nn.Conv2d):  hooks.append(m.register_forward_hook(conv_hook))
        if isinstance(m, nn.Linear):  hooks.append(m.register_forward_hook(lin_hook))
    net.eval()
    with torch.no_grad(): net(x)
    for h in hooks: h.remove()
    return flops, params

# ===================================================================
# 2. Measure FLOPS as we dial phi -> does the multiplier track 2^phi?
# ===================================================================
base_net, base_res = build(phi=0.0)
base_flops, base_params = count(base_net, base_res)
print("\nbaseline (phi=0):  res=%d  FLOPS=%.2fM  params=%.2fM"
      % (base_res, base_flops/1e6, base_params/1e6))

print("\nphi  res   FLOPS(M)  measured_x  predicted 2^phi")
for phi in [0, 1, 2, 3]:
    net, res = build(phi=phi)
    f, p = count(net, res)
    print("%2d  %4d   %7.2f    %6.2fx      %5.2f"
          % (phi, res, f/1e6, f/base_flops, 2**phi))
# measured ~ 1.00x, 1.92x, 4.38x, 8.24x  vs  predicted 1, 2, 4, 8 -> tracks ~2^phi.


# ===================================================================
# 3. ABLATION: same compute budget (phi=2 -> ~4x), compound vs single-dim.
# ===================================================================
print("\nABLATION at phi=2 (same ~4x FLOPS budget):")
for name, kw in [("compound  ", {}),
                 ("depth-only", {"depth_only": True}),
                 ("width-only", {"width_only": True}),
                 ("res-only  ", {"res_only":   True})]:
    net, res = build(phi=2, **kw)
    f, p = count(net, res)
    print("  %s  res=%3d  FLOPS=%6.2fM  params=%6.3fM"
          % (name, res, f/1e6, p/1e6))
# Same FLOPS budget, but compound spreads it across depth+width+resolution;
# the single-dim variants spend it all on one knob (note how params explode for
# width-only and how depth-only/res-only leave channels tiny). This is OUR small
# run -- the paper's accuracy comparison (Fig.8/Table 7) needs full ImageNet training.

## Visualize the data & results

_At a fixed compute (FLOPS) budget, how does compound scaling split the budget across depth/width/resolution vs pouring it all into one dimension?_

In [ ]:
import torch
import torch.nn as nn

# Reproduce the scaling arithmetic + FLOPS accounting on a toy 3-stage CNN.
ALPHA, BETA, GAMMA = 1.2, 1.1, 1.15
BASE_STAGES = [(24, 2), (48, 2), (96, 3)]
BASE_RES, PROD = 48, ALPHA * BETA**2 * GAMMA**2     # PROD = 1.9203...

def scale(phi): return ALPHA**phi, BETA**phi, GAMMA**phi
def round_ch(c, w): return max(8, int(round(c * w / 8)) * 8)

def build(phi, mode="compound"):
    d, w, r = scale(phi)
    if mode == "depth": d, w, r = PROD**phi, 1.0, 1.0
    if mode == "width": w = (PROD**phi)**0.5; d = r = 1.0
    if mode == "res":   r = (PROD**phi)**0.5; d = w = 1.0
    layers, in_ch = [], 3
    for ch, L in BASE_STAGES:
        out_ch  = round_ch(ch, w)
        n_layer = max(1, int(round(L * d)))
        for j in range(n_layer):
            stride = 2 if j == 0 else 1
            layers += [nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
                       nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
            in_ch = out_ch
    net = nn.Sequential(*layers, nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(in_ch, 10))
    return net, max(8, int(round(BASE_RES * r)))

def count(net, res):
    flops = [0]; params = sum(p.numel() for p in net.parameters()); hooks = []
    def ch(m, i, o): flops[0] += m.in_channels*m.out_channels*m.kernel_size[0]*m.kernel_size[1]*o.shape[2]*o.shape[3]
    def lh(m, i, o): flops[0] += m.in_features*m.out_features
    for m in net.modules():
        if isinstance(m, nn.Conv2d): hooks.append(m.register_forward_hook(ch))
        if isinstance(m, nn.Linear): hooks.append(m.register_forward_hook(lh))
    net.eval()
    with torch.no_grad(): net(torch.zeros(1, 3, res, res))
    for h in hooks: h.remove()
    return flops[0], params

bf, _ = count(*build(0))
print("FLOPS multiplier vs phi:")
for phi in [0, 1, 2, 3]:
    f, _ = count(*build(phi))
    print("  phi=%d  measured=%.2fx  predicted 2^phi=%d" % (phi, f/bf, 2**phi))

print("\nphi=2 budget split (params in M):")
for mode in ["compound", "depth", "width", "res"]:
    net, res = build(2, mode); f, p = count(net, res)
    print("  %-9s res=%d FLOPS=%.2fM params=%.3fM" % (mode, res, f/1e6, p/1e6))
# measured: phi=0->1.00x, 1->1.92x, 2->4.38x, 3->8.24x (tracks ~2^phi, base 1.92<2).
# params(M): compound 0.544, depth 1.042, width 0.918, res 0.246.
# Our small run -- cost/shape only; the paper's accuracy comparison needs ImageNet.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a fixed budget of about 2&times; the baseline FLOPS. Compare
            spending it as compound scaling ($\phi=1$ with the paper's $\alpha,\beta,\gamma$) versus pouring
            it all into depth. What are the depth/width/resolution multipliers in each case, and why does
            the compound split tend to win at equal cost?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compound, $\phi=1$: $d=1.2^1=1.2$, $w=1.1^1=1.1$, $r=1.15^1=1.15$. Check cost: $d\,w^2 r^2 = 1.2\times1.21\times1.3225 \approx 1.92 \approx 2$. — _One dial moves all three knobs in the balanced proportion the constraint enforces, landing right on the ~2&times; budget._
- Depth-only at the same ~2&times; budget: with $w=r=1$, the cost is just $d$, so $d\approx2$ (twice as many layers) and $w=r=1$. — _Depth is linear in FLOPS, so the whole 2&times; budget buys one knob: 2&times; depth, nothing for width or resolution._
- Reason about it: the depth-only net feeds the same tiny image into a much deeper stack; Observation 1 says one knob saturates, and Observation 2 says balance is critical. — _Extra layers can't extract detail the small input never had; the compound net adds resolution AND the depth/width to use it._

**Answer:** Compound ($\phi=1$): $d=1.2,\ w=1.1,\ r=1.15$ (every knob nudged up, total cost
                 $\approx2\times$). Depth-only: $d\approx2,\ w=1,\ r=1$ (all budget on layers). At the
                 same FLOPS the compound split usually wins because a single dimension saturates
                 (Observation 1) and balancing all three is what pays off (Observation 2): a deeper net on a
                 still-tiny image has little new signal to extract, while the compound net raises resolution and
                 adds the depth/width needed to exploit it. The paper's Fig. 8 / Table 7 shows up to ~2.5% higher
                 accuracy for compound at equal FLOPS. The CODEVIZ panel reproduces this cost-vs-split contrast on
                 a toy net.

</details>

**Problem 2.** Verify the balance constraint and the cost rule for the paper's coefficients. With
            $\alpha=1.2,\ \beta=1.1,\ \gamma=1.15$, (a) confirm $\alpha\cdot\beta^2\cdot\gamma^2\approx2$,
            and (b) compute the FLOPS multiplier at $\phi=3$ and compare it to $2^3$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Constraint: $\beta^2=1.21$, $\gamma^2=1.3225$, so $\alpha\beta^2\gamma^2 = 1.2\times1.21\times1.3225 \approx 1.920$. — _The grid search targets $\approx2$; $1.92$ is within the paper's tolerance._
- Cost at $\phi=3$: FLOPS multiplier $=(\alpha\beta^2\gamma^2)^\phi = 1.920^3 \approx 7.08$. — _From the derivation, total FLOPS scale by $(\alpha\beta^2\gamma^2)^\phi$, not by $\phi$ itself._
- Compare: $2^3 = 8$. The estimate $7.08$ is a bit under $8$ because the base is $1.92$, slightly below $2$. — _Holding the product near $2$ makes "$+1$ in $\phi$ ≈ double the FLOPS" hold approximately, not exactly._

**Answer:** (a) $\alpha\cdot\beta^2\cdot\gamma^2 = 1.2\times1.21\times1.3225 \approx 1.92 \approx 2$
                 ✓. (b) At $\phi=3$ the FLOPS multiplier is $1.92^3 \approx 7.1$, close to $2^3=8$. The small
                 shortfall is exactly because the base is $1.92$ rather than $2$. This is the "each $+1$ in $\phi$
                 roughly doubles the compute" rule, made concrete.

</details>

**Problem 3.** A teammate scales a CNN by setting $w=2$ (double the channels), leaving $d=r=1$, and is surprised the
            FLOPS go up about 4&times;, not 2&times;. Explain the factor of 4, and what the compound method
            would have done with that same ~4&times; budget.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Width touches both ends of every conv: input channels $\times2$ and output channels $\times2$. — _A conv connects every input channel to every output channel, so the channel-pair count is $C_{in}\cdot C_{out}$._
- So the FLOPS scale by $w^2 = 2^2 = 4$, not $w=2$. — _Width is quadratic in FLOPS — that $w^2$ term is why $\beta$ is squared in the constraint._
- Compound at the matching budget: $(\alpha\beta^2\gamma^2)^\phi \approx 2^\phi = 4$ means $\phi=2$, giving $d=1.44,\ w=1.21,\ r=1.32$. — _The same ~4&times; compute, but split across all three knobs in the balanced proportion._

**Answer:** Doubling width multiplies both the input- and output-channel counts, and a conv's cost
                 depends on their product $C_{in}\cdot C_{out}$, so FLOPS scale by $w^2 = 4$, not $2$. That $w^2$
                 is precisely why $\beta$ is squared in $\alpha\cdot\beta^2\cdot\gamma^2$. With the same
                 ~4&times; budget, compound scaling uses $\phi=2$: $d=1.44,\ w=1.21,\ r=1.32$ &mdash; a balanced
                 growth rather than 4&times; the compute poured into channels alone.

</details>